# Two-Tower Quantile Regressor — Cached Embeddings + Oligo Features
**Google Colab notebook**

Architecture from `AMP_twotower_quantile_colab.ipynb` adapted to use:
- **Peptide tower**: pre-computed ESM-2 mean-pool embeddings (cached `.npz`) → MLP
- **Organism tower**: oligonucleotide genome composition features → MLP
- **Fusion head**: `[p, o, p*o, |p-o|]` → MLP → quantile outputs (5%, 50%, 95%)

Data loading follows `genome_cnn_colab.ipynb` (no live ESM-2 inference needed).


In [ ]:
#@title 1. Setup & mount Drive
import os

# Mount Google Drive (assumes data is in Drive)
from google.colab import drive
drive.mount('/content/drive', force_remount=False)

# Set your data root (adjust path to match your Drive folder)
DATA_ROOT = "/content/drive/MyDrive/amp-prediction"  #@param {type:"string"}

import torch
import numpy as np
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Device: {DEVICE}")
print(f"Data root: {DATA_ROOT}")


In [ ]:
#@title 2. Verify data files
from pathlib import Path

required_files = {
    "data/raw/amp_mic_activities.csv": "Raw MIC data",
    "data/processed/embeddings/genome/oligo_composition.parquet": "Oligo genome features",
}

all_ok = True
for rel_path, desc in required_files.items():
    full_path = Path(DATA_ROOT) / rel_path
    exists = full_path.exists()
    status = "OK" if exists else "MISSING"
    if not exists:
        all_ok = False
    print(f"  [{status}] {desc}: {full_path}")

if not all_ok:
    print("\n  WARNING: Some files are missing. Upload them or adjust DATA_ROOT.")
else:
    print("\n  All files found!")


In [ ]:
#@title 3. Load & prepare data
import sys
from pathlib import Path
sys.path.insert(0, str(Path(DATA_ROOT)))

import numpy as np
import pandas as pd
import re

df = pd.read_csv(f"{DATA_ROOT}/data/raw/amp_mic_activities.csv")
print(f"Raw rows: {len(df):,}")

df = df.dropna(subset=["sequence", "target_activity_name", "activity"])
df["sequence"] = df["sequence"].str.upper().str.strip()
df["target_activity_name"] = df["target_activity_name"].str.strip()
df["activity"] = pd.to_numeric(df["activity"], errors="coerce")
df = df[df["activity"] > 0]
df = df[df["sequence"].str.len() > 0]
df = df[~df["sequence"].str.contains(r"[^ACDEFGHIKLMNPQRSTVWY]", flags=re.IGNORECASE)]
df["log_mic"] = np.log10(df["activity"])
df["species"] = df["target_activity_name"].apply(lambda x: " ".join(str(x).split()[:2]))

agg = df.groupby(["sequence", "species"], as_index=False).agg(
    log_mic=("log_mic", "mean"),
    n_measurements=("log_mic", "size"),
    target_activity_name=("target_activity_name", "first")
)
print(f"After aggregation: {len(agg):,} unique (peptide, species) pairs")
print(f"Unique peptides: {agg['sequence'].nunique():,}")
print(f"Unique species: {agg['species'].nunique():,}")


In [ ]:
#@title 4. Load oligonucleotide genome features
sys.path.insert(0, str(Path(DATA_ROOT)))
from src.features.genome import GenomeEncoder

genome_enc = GenomeEncoder()
oligo = pd.read_parquet(f"{DATA_ROOT}/data/processed/embeddings/genome/oligo_composition.parquet")
print(f"Oligo features: {oligo.shape[0]} species x {oligo.shape[1]-1} features")

oligo_cols = [c for c in oligo.columns if c != "species"]
agg = agg.merge(oligo, on="species", how="left")

matched = agg[oligo_cols[0]].notna().sum()
print(f"Genome match rate: {matched}/{len(agg)} ({matched/len(agg)*100:.1f}%)")
agg[oligo_cols] = agg[oligo_cols].fillna(0.0)


In [ ]:
#@title 5. Load or compute ESM-2 peptide embeddings
from pathlib import Path

ESM2_MODEL = "facebook/esm2_t12_35M_UR50D"
ESM2_DIM = 480
CACHE_PATH = Path(f"{DATA_ROOT}/data/processed/embeddings/facebook_esm2_t12_35M_UR50D_mic_embeddings.npz")

unique_seqs = sorted(agg["sequence"].unique())
print(f"Need embeddings for {len(unique_seqs)} unique sequences")

seq_to_emb = {}
if CACHE_PATH.exists():
    cache = np.load(CACHE_PATH, allow_pickle=True)
    for s, e in zip(cache["sequences"], cache["embeddings"]):
        seq_to_emb[s] = e
    print(f"Loaded {len(seq_to_emb)} embeddings from cache")

missing_seqs = [s for s in unique_seqs if s not in seq_to_emb]
print(f"Missing sequences: {len(missing_seqs)}")

if missing_seqs:
    from transformers import AutoTokenizer, AutoModel
    print(f"Computing ESM-2 embeddings for {len(missing_seqs)} sequences on {DEVICE}...")

    tokenizer = AutoTokenizer.from_pretrained(ESM2_MODEL)
    esm_model = AutoModel.from_pretrained(ESM2_MODEL).eval().to(DEVICE)

    batch_size = 32
    with torch.no_grad():
        for i in range(0, len(missing_seqs), batch_size):
            batch = missing_seqs[i:i+batch_size]
            inputs = tokenizer(
                batch, return_tensors="pt", padding=True,
                truncation=True, max_length=512,
                return_special_tokens_mask=True
            ).to(DEVICE)
            outputs = esm_model(**inputs)
            mask = inputs["attention_mask"].bool() & ~inputs["special_tokens_mask"].bool()
            mask_3d = mask.unsqueeze(-1).float()
            pooled = (outputs.last_hidden_state * mask_3d).sum(1) / mask_3d.sum(1).clamp(min=1)
            for seq, emb in zip(batch, pooled.cpu().numpy()):
                seq_to_emb[seq] = emb
            if (i // batch_size) % 50 == 0:
                print(f"  [{i+len(batch)}/{len(missing_seqs)}]")

    del esm_model
    torch.cuda.empty_cache()

    all_seqs = sorted(seq_to_emb.keys())
    CACHE_PATH.parent.mkdir(parents=True, exist_ok=True)
    np.savez(CACHE_PATH, sequences=all_seqs, embeddings=np.array([seq_to_emb[s] for s in all_seqs]))
    print(f"Updated cache: {len(all_seqs)} sequences")

peptide_embeddings = np.vstack([seq_to_emb[s] for s in agg["sequence"]])
print(f"Peptide embeddings: {peptide_embeddings.shape}")


In [ ]:
#@title 6. Train/validation split (grouped by peptide)
from sklearn.model_selection import GroupShuffleSplit

gss = GroupShuffleSplit(n_splits=1, test_size=0.15, random_state=42)
train_idx, val_idx = next(gss.split(agg, groups=agg["sequence"]))

print(f"Train: {len(train_idx):,} | Val: {len(val_idx):,}")

genome_features = agg[oligo_cols].values.astype(np.float32)

X_pep_train = peptide_embeddings[train_idx]
X_pep_val = peptide_embeddings[val_idx]
X_gen_train = genome_features[train_idx]
X_gen_val = genome_features[val_idx]
y_train = agg["log_mic"].values[train_idx].astype(np.float32)
y_val = agg["log_mic"].values[val_idx].astype(np.float32)

print(f"Feature dims: peptide={X_pep_train.shape[1]}, genome={X_gen_train.shape[1]}")


In [ ]:
#@title 7. Normalize features
from sklearn.preprocessing import StandardScaler

pep_scaler = StandardScaler()
gen_scaler = StandardScaler()

X_pep_train = pep_scaler.fit_transform(X_pep_train)
X_pep_val = pep_scaler.transform(X_pep_val)
X_gen_train = gen_scaler.fit_transform(X_gen_train)
X_gen_val = gen_scaler.transform(X_gen_val)

print(f"Peptide features normalized: mean={X_pep_train.mean():.4f}, std={X_pep_train.std():.4f}")
print(f"Genome features normalized: mean={X_gen_train.mean():.4f}, std={X_gen_train.std():.4f}")


In [ ]:
#@title 8. Define Two-Tower Quantile Model
import torch
import torch.nn as nn
import torch.nn.functional as F
from dataclasses import dataclass


@dataclass
class TwoTowerConfig:
    peptide_dim: int = 480
    genome_dim: int = 1344
    # tower hidden dims
    pep_hidden: int = 256
    org_hidden: int = 128
    # tower depth
    pep_layers: int = 2
    org_layers: int = 2
    # fusion
    shared_dim: int = 128
    head_hidden: int = 128
    # quantile regression
    head_type: str = "quantile"       # 'point' | 'quantile'
    quantiles: tuple = (0.05, 0.5, 0.95)
    dropout: float = 0.2


class PeptideTower(nn.Module):
    """MLP tower over pre-computed ESM-2 embeddings."""
    def __init__(self, cfg: TwoTowerConfig):
        super().__init__()
        layers = []
        in_dim = cfg.peptide_dim
        for _ in range(cfg.pep_layers):
            layers.extend([
                nn.Linear(in_dim, cfg.pep_hidden),
                nn.LayerNorm(cfg.pep_hidden),
                nn.GELU(),
                nn.Dropout(cfg.dropout),
            ])
            in_dim = cfg.pep_hidden
        self.net = nn.Sequential(*layers)

    def forward(self, x):
        return self.net(x)


class OrganismTower(nn.Module):
    """MLP tower over oligonucleotide genome features."""
    def __init__(self, cfg: TwoTowerConfig):
        super().__init__()
        layers = []
        in_dim = cfg.genome_dim
        for _ in range(cfg.org_layers):
            layers.extend([
                nn.Linear(in_dim, cfg.org_hidden),
                nn.LayerNorm(cfg.org_hidden),
                nn.GELU(),
                nn.Dropout(cfg.dropout),
            ])
            in_dim = cfg.org_hidden
        self.net = nn.Sequential(*layers)

    def forward(self, x):
        return self.net(x)


class TwoTowerMIC(nn.Module):
    """
    Two-tower model for (peptide_embedding, genome_features) -> MIC quantiles.

    Fusion: project both towers to shared_dim, form [p, o, p*o, |p-o|],
    then MLP head -> quantile outputs.
    """
    def __init__(self, cfg: TwoTowerConfig):
        super().__init__()
        self.cfg = cfg
        self.pep_tower = PeptideTower(cfg)
        self.org_tower = OrganismTower(cfg)

        self.pep_proj = nn.Linear(cfg.pep_hidden, cfg.shared_dim)
        self.org_proj = nn.Linear(cfg.org_hidden, cfg.shared_dim)

        fusion_in = cfg.shared_dim * 4  # [p, o, p*o, |p-o|]

        if cfg.head_type == "point":
            n_out = 1
        elif cfg.head_type == "quantile":
            n_out = len(cfg.quantiles)
        else:
            raise ValueError(f"Unknown head_type: {cfg.head_type}")

        self.head = nn.Sequential(
            nn.Linear(fusion_in, cfg.head_hidden),
            nn.GELU(),
            nn.Dropout(cfg.dropout),
            nn.Linear(cfg.head_hidden, cfg.head_hidden // 2),
            nn.GELU(),
            nn.Dropout(cfg.dropout),
            nn.Linear(cfg.head_hidden // 2, n_out),
        )
        self._init_weights()

    def _init_weights(self):
        for m in self.modules():
            if isinstance(m, nn.Linear):
                nn.init.kaiming_normal_(m.weight, nonlinearity="linear")
                if m.bias is not None:
                    nn.init.zeros_(m.bias)

    def forward(self, pep_features, gen_features):
        p = self.pep_tower(pep_features)
        o = self.org_tower(gen_features)
        p = self.pep_proj(p)
        o = self.org_proj(o)
        fusion = torch.cat([p, o, p * o, (p - o).abs()], dim=1)
        return self.head(fusion)

    def predict(self, out):
        """Convert raw output to point estimate + interval."""
        cfg = self.cfg
        if cfg.head_type == "point":
            return dict(mean=out[:, 0])
        qs = list(cfg.quantiles)
        d = {f"q{int(q*100)}": out[:, i] for i, q in enumerate(qs)}
        mid = qs.index(0.5) if 0.5 in qs else len(qs) // 2
        d["mean"] = out[:, mid]
        d["lower"] = out[:, 0]
        d["upper"] = out[:, -1]
        return d


# --- Losses ---
def pinball_loss(pred, target, quantiles):
    """Quantile (pinball) loss. pred: (B, n_q), target: (B,)."""
    target = target.unsqueeze(1)
    q = torch.tensor(quantiles, device=pred.device, dtype=pred.dtype).unsqueeze(0)
    e = target - pred
    loss = torch.maximum(q * e, (q - 1) * e)
    return loss.mean()


def huber_loss(pred, target, delta=1.0):
    return F.huber_loss(pred.squeeze(-1), target, delta=delta)


def compute_loss(cfg, out, target):
    if cfg.head_type == "point":
        return huber_loss(out, target, delta=1.0)
    elif cfg.head_type == "quantile":
        return pinball_loss(out, target, cfg.quantiles)
    raise ValueError(cfg.head_type)


# Instantiate
cfg = TwoTowerConfig(
    peptide_dim=X_pep_train.shape[1],
    genome_dim=X_gen_train.shape[1],
)
model = TwoTowerMIC(cfg).to(DEVICE)

n_params = sum(p.numel() for p in model.parameters())
print(f"Model parameters: {n_params:,}")
print(f"Head type: {cfg.head_type} | Quantiles: {cfg.quantiles}")
print(f"Architecture: PeptideTower({cfg.peptide_dim}->{cfg.pep_hidden}) + "
      f"OrganismTower({cfg.genome_dim}->{cfg.org_hidden}) -> fusion({cfg.shared_dim}*4) -> head")


In [ ]:
#@title 9. PyTorch Dataset & DataLoaders

class MICDataset(torch.utils.data.Dataset):
    def __init__(self, pep, gen, targets):
        self.pep = torch.tensor(pep, dtype=torch.float32)
        self.gen = torch.tensor(gen, dtype=torch.float32)
        self.targets = torch.tensor(targets, dtype=torch.float32)

    def __len__(self):
        return len(self.targets)

    def __getitem__(self, idx):
        return self.pep[idx], self.gen[idx], self.targets[idx]


BATCH_SIZE = 512
train_dataset = MICDataset(X_pep_train, X_gen_train, y_train)
val_dataset = MICDataset(X_pep_val, X_gen_val, y_val)

train_loader = torch.utils.data.DataLoader(
    train_dataset, batch_size=BATCH_SIZE, shuffle=True, num_workers=2, pin_memory=True
)
val_loader = torch.utils.data.DataLoader(
    val_dataset, batch_size=1024, shuffle=False, num_workers=2, pin_memory=True
)

print(f"Train batches: {len(train_loader)} | Val batches: {len(val_loader)}")


In [ ]:
#@title 10. Training loop
import time
from collections import defaultdict
from scipy.stats import spearmanr

MAX_EPOCHS = 300
PATIENCE = 30
LR = 3e-4
WEIGHT_DECAY = 2e-4

optimizer = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=WEIGHT_DECAY)
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(
    optimizer, T_max=MAX_EPOCHS, eta_min=1e-6
)

history = defaultdict(list)
best_val_spearman = -float("inf")
best_state = None
epochs_no_improve = 0

print(f"Training for up to {MAX_EPOCHS} epochs (patience={PATIENCE})")
print(f"Batch size: {BATCH_SIZE} | LR: {LR} | Head: {cfg.head_type}")
print("-" * 70)

for epoch in range(1, MAX_EPOCHS + 1):
    t0 = time.time()

    # --- Train ---
    model.train()
    train_loss_sum = 0.0
    train_n = 0
    for pep, gen, y in train_loader:
        pep, gen, y = pep.to(DEVICE), gen.to(DEVICE), y.to(DEVICE)
        out = model(pep, gen)
        loss = compute_loss(cfg, out, y)
        optimizer.zero_grad()
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        optimizer.step()
        train_loss_sum += loss.item() * len(y)
        train_n += len(y)

    scheduler.step()
    train_loss = train_loss_sum / train_n

    # --- Validate ---
    model.eval()
    val_preds, val_targets = [], []
    val_loss_sum = 0.0
    val_n = 0
    with torch.no_grad():
        for pep, gen, y in val_loader:
            pep, gen, y = pep.to(DEVICE), gen.to(DEVICE), y.to(DEVICE)
            out = model(pep, gen)
            val_loss_sum += compute_loss(cfg, out, y).item() * len(y)
            val_n += len(y)
            pred = model.predict(out)
            val_preds.append(pred["mean"].cpu())
            val_targets.append(y.cpu())

    val_loss = val_loss_sum / val_n
    val_preds_np = torch.cat(val_preds).numpy()
    val_targets_np = torch.cat(val_targets).numpy()

    val_mae = np.mean(np.abs(val_preds_np - val_targets_np))
    ss_res = np.sum((val_targets_np - val_preds_np)**2)
    ss_tot = np.sum((val_targets_np - val_targets_np.mean())**2)
    val_r2 = 1 - ss_res / ss_tot
    val_spearman = spearmanr(val_targets_np, val_preds_np).correlation

    elapsed = time.time() - t0
    history["train_loss"].append(train_loss)
    history["val_loss"].append(val_loss)
    history["val_mae"].append(val_mae)
    history["val_r2"].append(val_r2)
    history["val_spearman"].append(val_spearman)
    history["lr"].append(optimizer.param_groups[0]["lr"])

    improved = ""
    if val_spearman > best_val_spearman + 1e-4:
        best_val_spearman = val_spearman
        best_state = {k: v.cpu().clone() for k, v in model.state_dict().items()}
        epochs_no_improve = 0
        improved = " *"
    else:
        epochs_no_improve += 1

    if epoch <= 5 or epoch % 5 == 0 or improved:
        print(f"Epoch {epoch:3d} | train_loss={train_loss:.4f} | val_loss={val_loss:.4f} | "
              f"mae={val_mae:.4f} | r2={val_r2:.4f} | spearman={val_spearman:.4f} | "
              f"lr={optimizer.param_groups[0]['lr']:.2e} | {elapsed:.1f}s{improved}")

    if epochs_no_improve >= PATIENCE:
        print(f"\nEarly stopping at epoch {epoch} (best val_spearman={best_val_spearman:.4f})")
        break

if best_state:
    model.load_state_dict(best_state)
    model = model.to(DEVICE)
print(f"\nBest model: val_spearman={best_val_spearman:.4f}")


In [ ]:
#@title 11. Final evaluation & plots
import matplotlib.pyplot as plt
from scipy.stats import spearmanr, pearsonr

model.eval()
with torch.no_grad():
    final_preds = []
    final_lowers = []
    final_uppers = []
    for pep, gen, y in val_loader:
        pep, gen = pep.to(DEVICE), gen.to(DEVICE)
        out = model(pep, gen)
        pred = model.predict(out)
        final_preds.append(pred["mean"].cpu())
        final_lowers.append(pred["lower"].cpu())
        final_uppers.append(pred["upper"].cpu())

    final_preds = torch.cat(final_preds).numpy()
    final_lowers = torch.cat(final_lowers).numpy()
    final_uppers = torch.cat(final_uppers).numpy()

final_mae = np.mean(np.abs(final_preds - y_val))
final_rmse = np.sqrt(np.mean((final_preds - y_val)**2))
ss_res = np.sum((y_val - final_preds)**2)
ss_tot = np.sum((y_val - y_val.mean())**2)
final_r2 = 1 - ss_res / ss_tot
final_spearman = spearmanr(y_val, final_preds).correlation
final_pearson = pearsonr(y_val, final_preds)[0]

# Prediction interval coverage (PICP)
covered = ((y_val >= final_lowers) & (y_val <= final_uppers)).mean()
interval_width = np.mean(final_uppers - final_lowers)

print("=" * 55)
print("FINAL VALIDATION METRICS (Two-Tower Quantile)")
print("=" * 55)
print(f"  MAE:      {final_mae:.4f}")
print(f"  RMSE:     {final_rmse:.4f}")
print(f"  R2:       {final_r2:.4f}")
print(f"  Spearman: {final_spearman:.4f}")
print(f"  Pearson:  {final_pearson:.4f}")
print(f"  90% PI coverage (PICP): {covered:.3f}")
print(f"  Mean PI width (MPIW):   {interval_width:.3f}")
print("=" * 55)

fig, axes = plt.subplots(1, 4, figsize=(20, 5))

# Learning curves
axes[0].plot(history["train_loss"], label="Train loss", alpha=0.8)
axes[0].plot(history["val_loss"], label="Val loss", alpha=0.8)
axes[0].set_xlabel("Epoch"); axes[0].set_ylabel("Loss")
axes[0].set_title("Learning Curves"); axes[0].legend(); axes[0].grid(True, alpha=0.3)

# Pred vs actual
axes[1].scatter(y_val, final_preds, alpha=0.1, s=5)
lims = [min(y_val.min(), final_preds.min()), max(y_val.max(), final_preds.max())]
axes[1].plot(lims, lims, "r--", linewidth=1)
axes[1].set_xlabel("Actual log10(MIC)"); axes[1].set_ylabel("Predicted")
axes[1].set_title(f"Pred vs Actual (R2={final_r2:.3f})"); axes[1].grid(True, alpha=0.3)

# Spearman over training
axes[2].plot(history["val_spearman"], color="green", alpha=0.8)
axes[2].set_xlabel("Epoch"); axes[2].set_ylabel("Spearman rho")
axes[2].set_title(f"Val Spearman (best={best_val_spearman:.4f})"); axes[2].grid(True, alpha=0.3)

# Residuals
residuals = final_preds - y_val
axes[3].hist(residuals, bins=50, edgecolor="white", alpha=0.7)
axes[3].axvline(0, color="red", linestyle="--")
axes[3].set_xlabel("Residual"); axes[3].set_ylabel("Count")
axes[3].set_title(f"Residuals (MAE={final_mae:.3f})"); axes[3].grid(True, alpha=0.3)

plt.tight_layout()
from pathlib import Path
Path("results/figures").mkdir(parents=True, exist_ok=True)
plt.savefig("results/figures/twotower_quantile_cached.png", dpi=150, bbox_inches="tight")
plt.show()


In [ ]:
#@title 12. Save model checkpoint
from pathlib import Path

save_dir = Path(f"{DATA_ROOT}/results/checkpoints")
save_dir.mkdir(parents=True, exist_ok=True)
fig_dir = Path(f"{DATA_ROOT}/results/figures")
fig_dir.mkdir(parents=True, exist_ok=True)

checkpoint = {
    "model_state_dict": model.state_dict(),
    "config": cfg.__dict__,
    "pep_scaler_mean": pep_scaler.mean_,
    "pep_scaler_scale": pep_scaler.scale_,
    "gen_scaler_mean": gen_scaler.mean_,
    "gen_scaler_scale": gen_scaler.scale_,
    "metrics": {
        "best_val_spearman": best_val_spearman,
        "final_mae": final_mae,
        "final_rmse": final_rmse,
        "final_r2": final_r2,
        "final_spearman": final_spearman,
        "picp_90": covered,
        "mpiw": interval_width,
    },
    "history": dict(history),
}

ckpt_path = str(save_dir / "twotower_quantile_cached.pt")
torch.save(checkpoint, ckpt_path)
print(f"Saved: {ckpt_path}")

from google.colab import files
files.download(ckpt_path)
